In [1]:
import os

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from scraper import fetch_website_contents

load_dotenv()

True

In [2]:
GEMINI_MODEL = "gemini-2.5-flash"
GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

msg = [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "Hello!"}]
gemini = OpenAI(base_url=GEMINI_URL, api_key=GEMINI_API_KEY)
res = gemini.chat.completions.create(model=GEMINI_MODEL, messages=msg)
res.choices[0].message.content

'Hi there! How can I help you today?'

In [3]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [4]:
def summarize_website(model_obj: OpenAI, model: str, url: str):
    website_content = fetch_website_contents(url)
    user_prompt = user_prompt_prefix + website_content
    msg = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    res = model_obj.chat.completions.create(model=model, messages=msg)
    return res.choices[0].message.content

In [5]:
edwar_url = "https://edwarddonner.com"
summarize_website(gemini, GEMINI_MODEL, edwar_url)

'This is Edward Donner\'s personal ode to himself and all things LLM, where he very subtly mentions his *best-selling, top-rated* Udemy courses (with 900,000 enrollments, in case you missed it). When he\'s not busy being an AI co-founder or battling LLMs in an arena, he\'s apparently a "very amateur" electronic musician.\n\nHis "posts" are from the future, featuring resources for AI Coder, AI Builder, and AI Engineering MLOps courses, conveniently dated 2025 and 2026 – because why wait for events to happen when you can announce them early?'

In [6]:
def nice_summary(model_obj: OpenAI, url: str, model=GEMINI_MODEL):
    content = summarize_website(model_obj=model_obj, model=model, url=url)
    display(Markdown(content))

In [7]:
nice_summary(model_obj=gemini, url=edwar_url)

This website belongs to Edward Donner, a man who, much to his friends' dismay, won't stop talking about LLMs. He's turned his digital sermons into best-selling Udemy courses with nearly a million enrollments, proving that people *do* want to hear him drone on. When he's not busy being a co-founder/CTO of an AI startup (or selling old ones), he's dabbling in "very amateur" electronic music and pitting LLMs against each other in an arena of "diplomacy and deviousness."

As for news, Ed is apparently a time traveler. His "posts" page is filled with future dates, announcing AI Coder resources for February 2026, AI Builder resources for January 2026, and an MLOps track for September 2025. He also kindly provides guidance from May 2025 on the optimal order to tackle his AI courses. Clearly, he's way ahead of us mere mortals.

# OLLAMA SETUP

In [8]:
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_URL = f"{OLLAMA_BASE_URL}/v1"
OLLAMA_TOKEN = os.getenv("OLLAMA_TOKEN", "ollama")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3.5:4b")

In [9]:
requests.get(OLLAMA_BASE_URL).content

b'Ollama is running'

In [10]:
# Get a fun fact
ollama = OpenAI(base_url=OLLAMA_URL, api_key=OLLAMA_TOKEN)
res = ollama.chat.completions.create(model=OLLAMA_MODEL, messages=[{"role": "user", "content": "Tell me a fun fact"}])
res.choices[0].message.content

'Here’s a fun one: **Wombats produce cube-shaped poop.**\n\nScientists believe this unique shape helps them mark their territory on uneven ground without their droppings rolling away (and ruining their scent markers).'

In [11]:
nice_summary(ollama, model=OLLAMA_MODEL, url=edwar_url)

**Edward "Ed" Donner** apparently turned his annoyance that friends got fed up listening to him about LLMs for hours into a full-time AI career at Nebula.io and Udemy. It’s impressive that he somehow convinced over 900,000 people to enroll in his courses across 194 countries despite initially trying to convince himself to just write some code or make electronic music.

Currently, his blog is filled with posts dated as far out as February 2026 regarding AI Agents and Agentic Engineers. He seems confident enough to know how coding will change for an entire decade before it happens, but maybe that confidence stems from being a Managing Director at JPMorgan prior to the AI startup explosion.